# Chromatin PTR pseudocount fix
March 18, 2024

Previously, to correct for divide by zero issues, the minimum of the low of the 80/20 ratio was capped to 1, which resulted in less than 1.0 ratios. 

Rather, to handle divide by zeros, we will add 1 to the numerator and denominator of the ratio.

In this notebook, we will go through all deconvolution results and recompute the peak to trough ratio values.

Future deconvolution results will not have this bug.

In [6]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
import glob
from src.utils import mkdir_safe
from src.config import load_yl_rg1_vst_config
from cc_src.peak_to_trough import compute_ptr_f
from src.timer import Timer

timer = Timer()

outdir = 'output/deconvolve_combined_g_opt_2024_03_14/chromatin/'
file_paths = glob.glob(f'{outdir}/*_f_*.npy')

save_path = 'output/deconvolve_combined_g_opt_2024_03_14/chromatin_ptr_fix'

mkdir_safe(save_path)

# Load a dummy chromatin model so we can recompute the peak to trough ratio
config = load_yl_rg1_vst_config(1)

# For each deconvolved gene, load the f data and recompute the peak to trough ratio
i = 0
for path in file_paths:
    filename = path.split('/')[-1]

    loaded_f_data = np.load(path)
    ptr_res = compute_ptr_f(config, loaded_f_data)    
    newfilename = filename.replace('_f_', '_ptr_')
    
    np.save(f"{save_path}/{newfilename}", ptr_res)
    
    if i % 100 == 0:
        print(f"{i}/{len(file_paths)} - {timer.get_time()}")
        
    i += 1


Creating directory: output/deconvolve_combined_g_opt_2024_03_14/chromatin_ptr_fix...Directory exists. Skipping.
0/5575 - 00:00:00.77
